# Specify the Catalog


In [0]:
%sql
USE CATALOG brazilian_e_commerce;

In [0]:
%sql
DROP SCHEMA IF EXISTS gold CASCADE;

CREATE SCHEMA gold;

# Create Star schema for Visualisation in Power BI

## Create Fact table 

In [0]:
%sql
-- filter only this order items that are in orders 
CREATE OR REPLACE TABLE gold.fact_sales AS
SELECT i.order_id, i.order_item_id, i.product_id, i.seller_id, i.price, i.freight_value
FROM silver.siv_order_items i
INNER JOIN silver.siv_orders o ON i.order_id = o.order_id;

SELECT * FROM gold.fact_sales
LIMIT 5;

order_id,order_item_id,product_id,seller_id,price,freight_value
00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,58.9,13.29
00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,239.9,19.93
000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,199.0,17.87
00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,12.99,12.79
00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,199.9,18.14


## Create Dimensions table 

In [0]:
%sql
SELECT customer_id, count(customer_unique_id) AS total
FROM silver.siv_customers
GROUP BY customer_id
HAVING total > 1
LIMIT 5;
-- Unique id is an intern identifier for each customer, so we dont need it for this purpouse 

customer_id,total


### dim products

In [0]:
%sql
CREATE OR REPLACE TABLE gold.dim_products AS
SELECT * EXCEPT(`_ingested_at`, `_source_file`)
FROM silver.siv_products;

SELECT * FROM gold.dim_products
LIMIT 5;

product_id,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name
1e9e8ef04dbcff4541ed26657ea517e5,40,287,1,225,16.0,10.0,14.0,perfumery
3aa071139cb16b67ca9e5dea641aaa2f,44,276,1,1000,30.0,18.0,20.0,art
96bd76ec8810374ed1b65e291975717f,46,250,1,154,18.0,9.0,15.0,sports_leisure
cef67bcfe19066a932b7673e239eb23d,27,261,1,371,26.0,4.0,26.0,baby
9dc1a7de274444849c219cff195d0b71,37,402,4,625,20.0,17.0,13.0,housewares


### dim sellers

In [0]:
%sql
CREATE OR REPLACE TABLE gold.dim_sellers AS
SELECT * EXCEPT(`_ingested_at`, `_source_file`)
FROM silver.siv_sellers;

SELECT * FROM gold.dim_sellers
LIMIT 5;

seller_id,seller_zip_code_prefix,seller_city,seller_state
3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP
51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP


### prepare dim products

#### orders

In [0]:
%sql

CREATE OR REPLACE VIEW gold.orders AS
(
SELECT DISTINCT
    * EXCEPT(`order_status`, `_ingested_at`, `_source_file`),
    DATE_TRUNC('month', order_purchase_timestamp) AS purchase_year_month,
    MONTH(order_purchase_timestamp)               AS purchase_month,
    YEAR(order_purchase_timestamp)                AS purchase_year,
    DATEDIFF(order_delivered_customer_date, order_purchase_timestamp)AS days_to_delivery,
    DATEDIFF(order_delivered_customer_date, order_estimated_delivery_date)   AS delay_days
FROM silver.siv_orders
);

SELECT order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date,days_to_delivery,delay_days FROM gold.orders
LIMIT 5;

order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date,days_to_delivery,delay_days
2017-10-02T10:56:33.000Z,2017-10-10T21:25:13.000Z,2017-10-18T00:00:00.000Z,8,-8
2018-07-24T20:41:37.000Z,2018-08-07T15:27:45.000Z,2018-08-13T00:00:00.000Z,14,-6
2018-08-08T08:38:49.000Z,2018-08-17T18:06:29.000Z,2018-09-04T00:00:00.000Z,9,-18
2017-11-18T19:28:06.000Z,2017-12-02T00:28:42.000Z,2017-12-15T00:00:00.000Z,14,-13
2018-02-13T21:18:39.000Z,2018-02-16T18:17:02.000Z,2018-02-26T00:00:00.000Z,3,-10


#### payments

In [0]:
%sql
-- We flatten this table for better filtering in power BI 
CREATE OR REPLACE VIEW gold.payments AS
SELECT
    order_id,
    SUM(payment_value)     AS total_payment_value,
    COUNT(*)               AS payment_methods_count,
    max_by(payment_type, payment_value) AS main_payment_type,
    SUM(payment_installments) AS total_installments
FROM silver.siv_payments
GROUP BY order_id;


SELECT * FROM gold.payments
LIMIT 5;

order_id,total_payment_value,payment_methods_count,main_payment_type,total_installments
b81ef226f3fe1789b1e8b2acac839d17,99.33000183105469,1,credit_card,8
a9810da82917af2d9aefd1278f1dcfa0,24.389999389648438,1,credit_card,1
25e8ea4e93396b6fa0d3dd708e76c1bd,65.70999908447266,1,credit_card,1
ba78997921bbcdc1373bb41e913ab953,107.77999877929688,1,credit_card,8
42fdf880ba16b47b59251dd489d4441a,128.4499969482422,1,credit_card,2


#### reviews

In [0]:
%sql
-- We flatten this table for better filtering in power BI 

CREATE OR REPLACE VIEW gold.reviews AS
SELECT
    order_id,
    ROUND(AVG(review_score), 0)  AS avg_review_score,
    MAX(review_score)  AS max_review_score,
    COUNT(*)           AS reviews_count
FROM silver.siv_reviews
GROUP BY order_id;


SELECT * FROM gold.reviews
LIMIT 5;

order_id,avg_review_score,max_review_score,reviews_count
73fc7af87114b39712e6da79b0a377eb,4.0,4,1
a548910a1c6147796b98fdf73dbeba33,5.0,5,1
f9e4b658b201a9f2ecdecbb34bed034b,5.0,5,1
658677c97b385a9be170737859d3511b,5.0,5,1
8e6bfb81e283fa7e4f11123a3fb894f1,5.0,5,1


#### customers

In [0]:
%sql
CREATE OR REPLACE VIEW gold.customers AS
SELECT * EXCEPT(`customer_unique_id`,`_ingested_at`, `_source_file`)
FROM silver.siv_customers;

SELECT * FROM gold.customers
LIMIT 5;


customer_id,customer_zip_code_prefix,customer_city,customer_state
06b8999e2fba1a1fbc88172c00ba8bc7,14409,franca,SP
18955e83d337fd6b2def6b18a428ac77,9790,sao bernardo do campo,SP
4e7b3e00288586ebd08712fdd0374a03,1151,sao paulo,SP
b2b6027bc5c5109e529d4dc6358b12c3,8775,mogi das cruzes,SP
4f2d8ab171c80ec8364f7c12e35b23ad,13056,campinas,SP


### dim orders

In [0]:
%sql
CREATE OR REPLACE TABLE gold.dim_orders AS
SELECT
    o.order_id,
    o.order_approved_at,
    o.order_delivered_carrier_date,
    o.order_delivered_customer_date,
    o.order_estimated_delivery_date,
    o.order_purchase_timestamp,
    o.purchase_year_month,
    o.purchase_month,
    o.purchase_year,
    o.days_to_delivery,
    o.delay_days,

    -- z dim_customers
    c.customer_city,
    c.customer_state,
    c.customer_zip_code_prefix,
    

    -- z dim_payments
    p.main_payment_type,
    p.payment_methods_count,
    p.total_installments,
    p.total_payment_value,

    -- z dim_reviews
    r.avg_review_score,
    r.max_review_score,
    r.reviews_count

FROM gold.orders o
LEFT JOIN gold.customers c ON o.customer_id = c.customer_id
LEFT JOIN gold.payments  p ON o.order_id    = p.order_id
LEFT JOIN gold.reviews   r ON o.order_id    = r.order_id;

SELECT * FROM gold.dim_orders
LIMIT 5

order_id,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_purchase_timestamp,purchase_year_month,purchase_month,purchase_year,days_to_delivery,delay_days,customer_city,customer_state,customer_zip_code_prefix,main_payment_type,payment_methods_count,total_installments,total_payment_value,avg_review_score,max_review_score,reviews_count
a4591c265e18cb1dcee52889e2d8acc3,2017-07-09T22:10:13.000Z,2017-07-11T14:58:04.000Z,2017-07-26T10:57:55.000Z,2017-08-01T00:00:00.000Z,2017-07-09T21:57:05.000Z,2017-07-01T00:00:00.000Z,7,2017,17,-6,congonhinhas,PR,86320,credit_card,1,6,175.25999450683594,4.0,4,1
e1da8361c76cab67aa3588a1fbf1af54,2017-07-04T04:10:19.000Z,2017-07-04T14:52:46.000Z,2017-07-12T20:17:57.000Z,2017-07-25T00:00:00.000Z,2017-07-01T15:42:32.000Z,2017-07-01T00:00:00.000Z,7,2017,11,-13,ipatinga,MG,35162,boleto,1,1,253.39999389648438,2.0,2,1
5447160f15621601fa9ba2b94f4906b3,2017-11-21T12:40:55.000Z,2017-11-22T20:17:08.000Z,2017-11-28T20:33:38.000Z,2017-12-14T00:00:00.000Z,2017-11-21T12:08:32.000Z,2017-11-01T00:00:00.000Z,11,2017,7,-16,curitiba,PR,80530,credit_card,1,1,161.77000427246094,5.0,5,1
89165e6d4c28047da0adf13ac8b3e1b3,2018-07-18T13:50:14.000Z,2018-07-24T14:14:00.000Z,2018-07-26T18:26:37.000Z,2018-07-30T00:00:00.000Z,2018-07-18T13:39:54.000Z,2018-07-01T00:00:00.000Z,7,2018,8,-4,guarulhos,SP,7172,credit_card,1,1,118.3499984741211,5.0,5,1
02aec12849744e266394182861265286,2017-08-31T18:05:09.000Z,2017-09-04T19:22:51.000Z,2017-09-12T20:59:50.000Z,2017-09-27T00:00:00.000Z,2017-08-31T17:46:40.000Z,2017-08-01T00:00:00.000Z,8,2017,12,-15,dias d'avila,BA,42850,credit_card,1,4,210.63999938964844,5.0,5,1


### dim geilocation

In [0]:
%sql
CREATE OR REPLACE TABLE gold.dim_geolocation AS
SELECT 
    geolocation_zip_code_prefix,
    AVG(geolocation_lat)   AS lat,
    AVG(geolocation_lng)   AS lng,
    MAX(geolocation_city)  AS city,
    MAX(geolocation_state) AS state
FROM silver.siv_geolocation
GROUP BY geolocation_zip_code_prefix;

SELECT * FROM gold.dim_geolocation
LIMIT 5;

geolocation_zip_code_prefix,lat,lng,city,state
1003,-23.548993391149185,-46.635731416590076,são paulo,SP
1021,-23.543000428572945,-46.632143269414485,são paulo,SP
1044,-23.545598453945583,-46.64145278930664,são paulo,SP
1030,-23.5395490472967,-46.6330228285356,são paulo,SP
1108,-23.52723217010498,-46.630353927612305,sao paulo,SP


In [0]:
%sql 

SELECT 
geolocation_zip_code_prefix,
count(*) AS total
FROM gold.dim_geolocation
GROUP BY geolocation_zip_code_prefix
HAVING total > 1 

geolocation_zip_code_prefix,total


# Create One Big Table (OBT) for ML modeling to predict customer raview score 

## To make this kind of table, we need to flatten some other tables, like order_items, because every row needs to represent a specific order rather than an item. 

### SALES 

In [0]:
%sql
SELECT order_id, count(*) AS total
FROM silver.siv_order_items
GROUP BY order_id
ORDER BY total DESC
LIMIT 5

order_id,total
8272b63d03f5f79c56e9e4120aec44ef,21
1b15974a0141d54e36626dca3fdc731a,20
ab14fdcfbe524636d65ee38360e22ce8,20
428a2f660dc84138d969ccd69a0ab6d5,15
9ef13efd6949e4573a18964dd1bbe7f5,15


In [0]:
%sql
SELECT 
i.* EXCEPT(seller_id,shipping_limit_date,_source_file,_ingested_at),
s.* EXCEPT(seller_id,_source_file,_ingested_at),
p.* EXCEPT(product_id,_source_file,_ingested_at)

FROM silver.siv_order_items as i 
LEFT JOIN silver.siv_sellers as s ON i.seller_id = s.seller_id
LEFT JOIN silver.siv_products as p on i.product_id = p.product_id
WHERE order_id = "0362e923f805ae4dce58fd78c86c96c4"
ORDER BY price DESC

order_id,order_item_id,product_id,price,freight_value,seller_zip_code_prefix,seller_city,seller_state,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name
0362e923f805ae4dce58fd78c86c96c4,1,8af4a717d0442a5fbe332dc587583296,129.9,71.46,36502,uba,MG,17,540,1,17500,97.0,24.0,45.0,kitchen_dining_laundry_garden_furniture
0362e923f805ae4dce58fd78c86c96c4,2,5f82c0dd37c20e3f5010be5f3c6a9ff1,40.85,71.46,13920,pedreira,SP,50,365,1,21400,48.0,65.0,48.0,housewares


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW joined_items AS
(
SELECT 
  i.order_id,
  i.order_item_id,
  i.product_id,
  i.seller_id,
  COALESCE(i.price, 0.0) AS price,
  COALESCE(i.freight_value, 0.0) AS freight_value,
  
  COALESCE(s.seller_zip_code_prefix, 'None') AS seller_zip_code_prefix,
  COALESCE(s.seller_city, 'None') AS seller_city,
  COALESCE(s.seller_state, 'None') AS seller_state,

  COALESCE(p.product_category_name, 'None_cat') AS product_category_name,
  COALESCE(p.product_name_lenght, 0) AS product_name_lenght,
  COALESCE(p.product_description_lenght, 0) AS product_description_lenght,
  COALESCE(p.product_photos_qty, 0) AS product_photos_qty,
  COALESCE(p.product_weight_g, 0.0) AS product_weight_g,
  COALESCE(p.product_length_cm, 0.0) AS product_length_cm,
  COALESCE(p.product_height_cm, 0.0) AS product_height_cm,
  COALESCE(p.product_width_cm, 0.0) AS product_width_cm

FROM silver.siv_order_items as i 
JOIN silver.siv_sellers as s ON i.seller_id = s.seller_id
JOIN silver.siv_products as p on i.product_id = p.product_id
);


CREATE OR REPLACE TEMP VIEW flaten_sales AS
(
  SELECT 
  order_id,
    count(order_item_id) AS total_items,
    sum(price) AS total_price,
    sum(freight_value) AS total_freight_value,
    
    max_by(product_category_name, product_weight_g) AS main_product_category, -- Category will be the most havy product in this order 
    
    max(seller_zip_code_prefix) AS main_seller_zip,
    max(seller_city) AS main_seller_city,
    max(seller_state) AS main_seller_state,
    
    avg(product_name_lenght) AS avg_product_name_lenght,
    avg(product_description_lenght) AS avg_product_description_lenght,
    avg(product_photos_qty) AS avg_product_photos_qty,
    avg(product_weight_g) AS avg_product_weight_g,
    avg(product_length_cm) AS avg_product_length_cm,
    avg(product_height_cm) AS avg_product_height_cm,
    avg(product_width_cm) AS avg_product_width_cm
FROM joined_items 
GROUP BY order_id
ORDER BY total_price DESC
)




In [0]:
%sql
SELECT *
FROM flaten_sales
WHERE order_id = "0362e923f805ae4dce58fd78c86c96c4"

order_id,total_items,total_price,total_freight_value,main_product_category,main_seller_zip,main_seller_city,main_seller_state,avg_product_name_lenght,avg_product_description_lenght,avg_product_photos_qty,avg_product_weight_g,avg_product_length_cm,avg_product_height_cm,avg_product_width_cm
0362e923f805ae4dce58fd78c86c96c4,2,170.74999237060547,142.9199981689453,housewares,36502,uba,SP,33.5,452.5,1.0,19450.00000,72.5,44.5,46.5


### ORDERS

In [0]:
%sql

SELECT *
from silver.siv_payments
WHERE order_id = "ad21c59c0840e6cb83a9ceb5573f8159"


order_id,payment_sequential,payment_type,payment_installments,payment_value,_ingested_at,_source_file
ad21c59c0840e6cb83a9ceb5573f8159,1,credit_card,1,28.62,2026-04-17T22:55:25.451Z,dbfs:/Volumes/brazilian_e_commerce/row_data/row/olist_order_payments_dataset.csv


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW transformed_orders AS
(
SELECT 
  order_id,
  customer_id,
  datediff(order_delivered_customer_date, order_purchase_timestamp) AS days_to_delivery,
  datediff(order_delivered_customer_date,order_estimated_delivery_date ) AS delivery_delay_days,
  datediff(order_approved_at, order_purchase_timestamp) AS payment_approval_days,
  datediff(order_delivered_carrier_date, order_approved_at) AS seller_processing_days,
  datediff(order_delivered_customer_date, order_delivered_carrier_date) AS carrier_transit_days,
  datediff(order_estimated_delivery_date, order_purchase_timestamp) AS promised_delivery_days,
  hour(order_purchase_timestamp) AS purchase_hour,    
  dayofweek(order_purchase_timestamp) AS purchase_day_of_week,       
  month(order_purchase_timestamp) AS purchase_month,
  year(order_purchase_timestamp) AS purchase_year
FROM silver.siv_orders
);

CREATE OR REPLACE TEMP VIEW transformed_customers AS
(
  SELECT
  customer_id,
  customer_zip_code_prefix,
  customer_city,
  customer_state
  FROM silver.siv_customers
);

CREATE OR REPLACE TEMP VIEW flaten_reviews AS
(
  SELECT order_id,
  count(*) AS total_count_reviews,
  max(review_score) AS max_review_score,
  avg(review_score) AS avg_review_score
  FROM silver.siv_reviews
  GROUP BY order_id
);

CREATE OR REPLACE TEMP VIEW flaten_payments AS
(
  SELECT order_id,
  max(payment_value) AS max_payment_value,
  avg(payment_value) AS avg_payment_value,
  max_by(payment_type, payment_value) AS main_payment_type,
  count(*) AS total_payments
  FROM silver.siv_payments
  GROUP BY order_id
);

CREATE OR REPLACE TEMP VIEW joined_orders AS
(
SELECT 
o.* EXCEPT(customer_id),
c.* EXCEPT(customer_id),

COALESCE(p.max_payment_value, 0.0) AS max_payment_value,
COALESCE(p.avg_payment_value, 0.0) AS avg_payment_value,
COALESCE(p.main_payment_type, 'brak_danych') AS main_payment_type,
COALESCE(p.total_payments, 0) AS total_payments,

COALESCE(r.total_count_reviews, 0) AS total_count_reviews,
COALESCE(r.max_review_score, -1) AS max_review_score,
COALESCE(r.avg_review_score, -1.0) AS avg_review_score

FROM transformed_orders AS o 
LEFT JOIN transformed_customers AS c ON o.customer_id = c.customer_id 
LEFT JOIN flaten_reviews AS r ON o.order_id = r.order_id 
LEFT JOIN flaten_payments AS p ON o.order_id = p.order_id
);


SELECT *
FROM joined_orders
LIMIT 5



order_id,days_to_delivery,delivery_delay_days,payment_approval_days,seller_processing_days,carrier_transit_days,promised_delivery_days,purchase_hour,purchase_day_of_week,purchase_month,purchase_year,customer_zip_code_prefix,customer_city,customer_state,max_payment_value,avg_payment_value,main_payment_type,total_payments,total_count_reviews,max_review_score,avg_review_score
e481f51cbdc54678b7cc49136f2d6af7,8,-8,0,2,6,16,10,2,10,2017,3149,sao paulo,SP,18.59000015258789,12.90333366394043,voucher,3,1,4,4.0
53cdb2fc8bc7dce0b6741e2150273451,14,-6,2,0,12,20,20,3,7,2018,47813,barreiras,BA,141.4600067138672,141.4600067138672,boleto,1,1,4,4.0
47770eb9100c2d0c44946d9cf07ec65d,9,-18,0,0,9,27,8,4,8,2018,75265,vianopolis,GO,179.1199951171875,179.1199951171875,credit_card,1,1,5,5.0
949d5b44dbf5de918fe9c16f97b45f8a,14,-13,0,4,10,27,19,7,11,2017,59296,sao goncalo do amarante,RN,72.19999694824219,72.19999694824219,credit_card,1,1,5,5.0
ad21c59c0840e6cb83a9ceb5573f8159,3,-10,0,1,2,13,21,3,2,2018,9195,santo andre,SP,28.6200008392334,28.6200008392334,credit_card,1,1,5,5.0


## join orders AND items

In [0]:
%sql 

CREATE OR REPLACE TABLE gold.OBT_ML AS
(
SELECT 
s.*,
o.* EXCEPT(order_id)

FROM joined_orders AS o
JOIN flaten_sales AS s
ON o.order_id = s.order_id
);

SELECT * FROM gold.OBT_ML
LIMIT 5

order_id,total_items,total_price,total_freight_value,main_product_category,main_seller_zip,main_seller_city,main_seller_state,avg_product_name_lenght,avg_product_description_lenght,avg_product_photos_qty,avg_product_weight_g,avg_product_length_cm,avg_product_height_cm,avg_product_width_cm,days_to_delivery,delivery_delay_days,payment_approval_days,seller_processing_days,carrier_transit_days,promised_delivery_days,purchase_hour,purchase_day_of_week,purchase_month,purchase_year,customer_zip_code_prefix,customer_city,customer_state,max_payment_value,avg_payment_value,main_payment_type,total_payments,total_count_reviews,max_review_score,avg_review_score
e481f51cbdc54678b7cc49136f2d6af7,1,29.989999771118164,8.720000267028809,housewares,9350,maua,SP,40.0,268.0,4.0,500.00000,19.0,8.0,13.0,8,-8,0,2,6,16,10,2,10,2017,3149,sao paulo,SP,18.59000015258789,12.90333366394043,voucher,3,1,4,4.0
53cdb2fc8bc7dce0b6741e2150273451,1,118.69999694824219,22.760000228881836,perfumery,31570,belo horizonte,SP,29.0,178.0,1.0,400.00000,19.0,13.0,19.0,14,-6,2,0,12,20,20,3,7,2018,47813,barreiras,BA,141.4600067138672,141.4600067138672,boleto,1,1,4,4.0
47770eb9100c2d0c44946d9cf07ec65d,1,159.89999389648438,19.219999313354492,auto,14840,guariba,SP,46.0,232.0,1.0,420.00000,24.0,19.0,21.0,9,-18,0,0,9,27,8,4,8,2018,75265,vianopolis,GO,179.1199951171875,179.1199951171875,credit_card,1,1,5,5.0
949d5b44dbf5de918fe9c16f97b45f8a,1,45.0,27.200000762939453,pet_shop,31842,belo horizonte,MG,59.0,468.0,3.0,450.00000,30.0,10.0,20.0,14,-13,0,4,10,27,19,7,11,2017,59296,sao goncalo do amarante,RN,72.19999694824219,72.19999694824219,credit_card,1,1,5,5.0
ad21c59c0840e6cb83a9ceb5573f8159,1,19.899999618530273,8.720000267028809,stationery,8752,mogi das cruzes,SP,38.0,316.0,4.0,250.00000,51.0,15.0,15.0,3,-10,0,1,2,13,21,3,2,2018,9195,santo andre,SP,28.6200008392334,28.6200008392334,credit_card,1,1,5,5.0


### LAST test of flattening  IF there more rows for one order_id and if there are any NULL Values 

In [0]:
%sql 

SELECT 
  order_id,
  count(*) AS total_items
FROM gold.OBT_ML
GROUP BY order_id
HAVING total_items > 1

order_id,total_items


In [0]:
%sql


SELECT *
FROM gold.OBT_ML
WHERE order_id IS NULL
OR total_items IS NULL
OR total_price IS NULL
OR max_payment_value IS NULL
OR avg_payment_value IS NULL
OR main_payment_type IS NULL
OR total_payments IS NULL
OR total_count_reviews IS NULL
OR max_review_score IS NULL
OR avg_review_score IS NULL
OR customer_zip_code_prefix IS NULL
OR customer_city IS NULL
OR customer_state IS NULL
OR days_to_delivery IS NULL
OR delivery_delay_days IS NULL
OR payment_approval_days IS NULL
OR seller_processing_days IS NULL
OR carrier_transit_days IS NULL
OR promised_delivery_days IS NULL
OR purchase_hour IS NULL
OR purchase_day_of_week IS NULL
OR purchase_month IS NULL
OR purchase_year IS NULL
OR avg_product_length_cm IS NULL
OR avg_product_height_cm IS NULL
OR avg_product_weight_g IS NULL
OR avg_product_width_cm IS NULL
OR avg_product_photos_qty IS NULL
OR avg_product_name_lenght IS NULL
OR avg_product_description_lenght IS NULL




order_id,total_items,total_price,total_freight_value,main_product_category,main_seller_zip,main_seller_city,main_seller_state,avg_product_name_lenght,avg_product_description_lenght,avg_product_photos_qty,avg_product_weight_g,avg_product_length_cm,avg_product_height_cm,avg_product_width_cm,days_to_delivery,delivery_delay_days,payment_approval_days,seller_processing_days,carrier_transit_days,promised_delivery_days,purchase_hour,purchase_day_of_week,purchase_month,purchase_year,customer_zip_code_prefix,customer_city,customer_state,max_payment_value,avg_payment_value,main_payment_type,total_payments,total_count_reviews,max_review_score,avg_review_score
